# llm-finetune-serve — Colab driver

This notebook stays thin on purpose: clone, install, call scripts. All logic
lives in `src/`. If you find yourself writing real code here, it belongs in the
repo instead.

## Two ways to run it

**In the browser:** open this notebook from GitHub, then Runtime → Change
runtime type → GPU.

**From VS Code** (no browser tab): install the official **Google Colab**
extension (publisher: Google), open this file locally, then kernel picker →
`Colab` → `Auto Connect`, and pick a GPU runtime.

Either way the kernel runs on a Colab VM, and the extension does **not** sync
local files to it — so your `src/` edits reach the GPU through GitHub. The loop
is: edit locally → commit + push → re-run the `git pull` cell below → re-run
the stage cell.

In [1]:
!nvidia-smi

Fri Sep  4 22:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Clone the repo

Public repo, so no credentials are needed. Re-running this cell pulls the
latest commit rather than re-cloning.

(If you ever flip it back to private, add a GitHub PAT with `repo` scope as a
Colab secret named `GITHUB_TOKEN` — the cell picks it up automatically.)

In [8]:
OWNER = "rushilpatra"
REPO = "llm-finetune-serve"
BRANCH = "main"
REPO_DIR = f"/content/{REPO}"

import os, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")

auth = f"{TOKEN}@" if TOKEN else ""
url = f"https://{auth}github.com/{OWNER}/{REPO}.git"

# Absolute path: a relative one would clone a second copy inside the first
# whenever this cell is re-run after the %cd below.
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
%cd /content/llm-finetune-serve
!git pull --ff-only

/content/llm-finetune-serve
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 14.86 KiB | 3.71 MiB/s, done.
From https://github.com/rushilpatra/llm-finetune-serve
   d3091b8..86e6177  main       -> origin/main
Updating d3091b8..86e6177
Fast-forward
 notebooks/run.ipynb | 639 ++++++++++++++++++++++++++++++++++++++++++++--------
 src/evaluate.py     |  24 +-
 2 files changed, 560 insertions(+), 103 deletions(-)


## 2. Install

Colab preinstalls `torch` / `torchvision` / `torchaudio` built against one CUDA
version, and vLLM pulls a torch built against another. Mixing them raises

    RuntimeError: Detected that PyTorch and TorchAudio were compiled with
    different CUDA versions

on `import vllm`. So we uninstall all four — vLLM included, otherwise pip sees
it already installed, skips it, and never reinstalls the torch we just removed
— and let a clean vLLM install pull a matched set.

**This replaces torch, so the kernel must be restarted afterwards.** In the
browser Colab prompts you; **in VS Code it does not** — click the **↺ Restart**
button in the notebook toolbar yourself. Then re-run the clone cell above and
skip straight to the version check; the install is cached.

In [12]:
# vLLM owns the torch stack. vLLM is uninstalled too, so pip actually
# re-resolves it instead of treating the requirement as already satisfied.
!pip uninstall -q -y vllm torch torchvision torchaudio
!pip install -q vllm
!pip install -q -r requirements.txt

In [2]:
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    # T4 (Turing) has no bf16 support; scripts detect this at runtime.
    print("gpu         ", torch.cuda.get_device_name(0))
    print("bf16        ", torch.cuda.is_bf16_supported())

# Import vLLM here rather than discovering a broken install 20 minutes into a run.
import vllm
print("vllm        ", vllm.__version__)

torch        2.13.0+cu130
transformers 5.16.1
cuda         True
gpu          NVIDIA A100-SXM4-40GB
bf16         True
vllm         0.28.0


## 3. Data smoke test

Prints split sizes, the 8-shot prefix length, and a sample prompt with the
round-trip answer-extraction check.

In [3]:
!python -m src.data --split val --limit 2

split sizes:
  fewshot  8
  val      750
  train    6715
  test     1319

8-shot prefix: 3600 chars

re male. If there are 18 contestants in total, how many of them are male?
Answer: There are 18/3 = 6 female contestants.
There are 18-6 = 12 male contestants.
#### 12

Question: Nancy bought a pie sliced it into 8 pieces. She gave 1/2 to Joe and Darcy, and she gave 1/4 to Carl. How many slices were left?
Answer: The total number of slices she gave to Joe and Darcy is 1/2 x 8 = 4.
The total slice she gave to Carl is 1/4 x 8 = 2.
Therefore, the total slices left is 8 - 4 - 2 = 2.
#### 2

Question: Megan pays $16 for a shirt that costs $22 before sales. What is the amount of the discount?
Answer:

[gold] 6
[extracted from gold completion] 6
[well formed] True
### 12

Question: Nancy bought a pie sliced it into 8 pieces. She gave 1/2 to Joe and Darcy, and she gave 1/4 to Carl. How many slices were left?
Answer: The total number of slices she gave to Joe and Darcy is 1/2 x 8 = 4.
The total s

## 4. Baseline: 8-shot prompting of the base model

Plumbing check first (no GPU, seconds), then the real run. Predictions stream
to `results/baseline_8shot.jsonl` as they are produced, so if the session dies
you re-run the same command and it picks up where it stopped.

In [10]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --dry-run --limit 4

!! DRY RUN: generations are gold completions, metrics are meaningless

4 examples, 4 already done, 0 to generate

run               baseline_8shot  (val, n=4)
exact match       1.0000 +/- 0.0000
format adherence  1.0000 +/- 0.0000
wall clock        0.3s
gpu               NVIDIA A100-SXM4-40GB  peak 0.45 GB

predictions  results/baseline_8shot-dryrun.jsonl
metrics      results/baseline_8shot-dryrun.metrics.json


In [11]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml

750 examples, 0 already done, 750 to generate
INFO 09-04 23:06:06 [api_utils.py:272] non-default args: {'dtype': 'bfloat16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-0.6B-Base'}
INFO 09-04 23:06:07 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-04 23:06:07 [model.py:1965] Using max model len 2048
INFO 09-04 23:06:07 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-04 23:06:07 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 09-04 23:06:12 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=15357) INFO 09-04 23:06:26 [core.py:122] Initializing a V1 L

## 5. Save results off the VM

Colab VMs are ephemeral, and `files.download()` only works from the Colab
browser UI — from a VS Code kernel it silently does nothing. So push `results/`
back to GitHub instead, which versions the predictions alongside the code.

You need a GitHub token with write access to the repo: **Settings → Developer
settings → Personal access tokens → Fine-grained**, repository access limited to
`llm-finetune-serve`, permission **Contents: Read and write**. Either paste it
when prompted, or store it once as a Colab secret named `GITHUB_TOKEN`.

In [13]:
import subprocess
from getpass import getpass

REPO_DIR = "/content/llm-finetune-serve"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass("GitHub PAT (Contents: Read and write): ").strip()


def git(*args):
    result = subprocess.run(
        ["git", *args], cwd=REPO_DIR, capture_output=True, text=True
    )
    # Redact: git echoes the remote URL, token and all, on both success and error.
    print((result.stdout + result.stderr).replace(TOKEN, "***").strip())
    return result.returncode


git("config", "user.name", "rushilpatra")
git("config", "user.email", "patra.rushil@gmail.com")
git("add", "results")
git("commit", "-m", "Add eval results from Colab")
git("push", f"https://{TOKEN}@github.com/rushilpatra/llm-finetune-serve.git", "HEAD:main")




On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	llm-finetune-serve/

nothing added to commit but untracked files present (use "git add" to track)
remote: Permission to rushilpatra/llm-finetune-serve.git denied to rushilpatra.
fatal: unable to access 'https://github.com/rushilpatra/llm-finetune-serve.git/': The requested URL returned error: 403


128

## Next stages

Cells for training, merging, and benchmarking get added here as those scripts
land. Each is a single `!python -m src.<script> --config configs/<run>.yaml`.